# Abgabe 4 – RANSAC und automatische Bildregistrierung
**Computer Vision – Prof. Dr. Matthias O. Franz**

Automatische Homographie-Schätzung aus SIFT-Korrespondenzen mittels RANSAC und normierter DLT.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
import os
import sys
import random

In [ ]:
def normalize_points(points):
    """Hartley-Normalisierung: Schwerpunkt -> 0, mittlere Distanz -> sqrt(2)."""
    points = np.array(points, dtype=float)
    centroid = np.mean(points, axis=0)
    points_centered = points - centroid
    scale = np.sqrt(2) / np.mean(np.linalg.norm(points_centered, axis=1))
    T = np.array([
        [scale, 0,    -scale * centroid[0]],
        [0,    scale, -scale * centroid[1]],
        [0,    0,      1]
    ])
    points_h = np.hstack([points, np.ones((len(points), 1))])
    points_norm = (T @ points_h.T).T
    return points_norm[:, :2], T


def projective_transform(image, H, output_shape):
    """Inverse Abbildung (Nearest-Neighbor). Gibt (result, mask) zurück."""
    h_out, w_out = output_shape
    squeezed = False
    if image.ndim == 2:
        image = image[:, :, np.newaxis]
        squeezed = True

    result = np.zeros((h_out, w_out, image.shape[2]), dtype=image.dtype)
    mask   = np.zeros((h_out, w_out), dtype=np.float32)
    H_inv  = np.linalg.inv(H)

    ys, xs = np.mgrid[0:h_out, 0:w_out]
    coords_h = np.stack([xs.ravel(), ys.ravel(), np.ones(h_out * w_out)])
    src_h = H_inv @ coords_h
    src_h /= src_h[2]
    x_src = src_h[0].reshape(h_out, w_out)
    y_src = src_h[1].reshape(h_out, w_out)

    x_int = np.round(x_src).astype(int)
    y_int = np.round(y_src).astype(int)
    valid = (x_int >= 0) & (x_int < image.shape[1]) &             (y_int >= 0) & (y_int < image.shape[0])

    result[valid] = image[y_int[valid], x_int[valid]]
    mask[valid]   = 1.0

    if squeezed:
        result = result[:, :, 0]
    return result, mask

## Teilaufgabe 1a – Direct Linear Transform (DLT)

Der **DLT** berechnet eine Homographie aus mindestens 4 Punktkorrespondenzen via SVD.

Für jede Korrespondenz $(x,y) \leftrightarrow (x',y')$ gilt $\mathbf{x}' \sim H\,\mathbf{x}$.  
Die Kreuzprodukt-Bedingung liefert 2 lineare Gleichungen pro Punkt → Designmatrix $A$ (2n×9).  
Die Lösung ist der letzte rechte Singulärvektor der SVD von $A$, umgeformt zu $H$ (3×3), normiert auf $H_{2,2}=1$.

**Normierter DLT (Hartley):** Quell- und Zielpunkte werden vor der DLT-Berechnung isotropisch normiert  
(Schwerpunkt → Ursprung, mittlere Distanz → $\sqrt{2}$). Danach Entnormierung: $H = T_2^{-1}\,\hat{H}\,T_1$.  
Das verbessert die numerische Konditionierung bei großen Pixelkoordinaten erheblich.

In [ ]:
def dlt(src_pts, dst_pts):
    """Unnormalisierter DLT. Gibt H (3x3) zurück, normiert auf H[2,2]=1."""
    src_pts = np.array(src_pts, dtype=float)
    dst_pts = np.array(dst_pts, dtype=float)
    assert len(src_pts) >= 4

    rows = []
    for (x, y), (xp, yp) in zip(src_pts, dst_pts):
        rows.append([ x,  y,  1,  0,  0,  0, -xp*x, -xp*y, -xp])
        rows.append([ 0,  0,  0,  x,  y,  1, -yp*x, -yp*y, -yp])

    A = np.array(rows)
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    return H / H[2, 2]


def normalized_dlt(src_pts, dst_pts):
    """Normierter DLT (Hartley). Gibt H (3x3) zurück, normiert auf H[2,2]=1."""
    src_pts = np.array(src_pts, dtype=float)
    dst_pts = np.array(dst_pts, dtype=float)

    src_norm, T1 = normalize_points(src_pts)
    dst_norm, T2 = normalize_points(dst_pts)

    H_hat = dlt(src_norm, dst_norm)
    H = np.linalg.inv(T2) @ H_hat @ T1
    return H / H[2, 2]

In [ ]:
print("=" * 60)
print("TEST 1: Identität – src == dst  →  H ≈ I")
pts = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])
H = dlt(pts, pts)
assert np.allclose(H, np.eye(3), atol=1e-6), f"H ≠ I:\n{H}"
print(f"  ✓ unnorm. DLT  H[0,0]={H[0,0]:.6f}")
H_n = normalized_dlt(pts, pts)
assert np.allclose(H_n, np.eye(3), atol=1e-6)
print(f"  ✓ norm.   DLT  H[0,0]={H_n[0,0]:.6f}")

print()
print("TEST 2: Bekannte Translation (+10, +5)")
src = np.array([[0.,0.],[100.,0.],[100.,100.],[0.,100.]])
dst = src + np.array([10., 5.])
for label, fn in [("unnorm.", dlt), ("norm.  ", normalized_dlt)]:
    H = fn(src, dst)
    p = H @ np.array([0., 0., 1.]); p /= p[2]
    assert np.allclose(p[:2], [10., 5.], atol=0.5), f"{label}: {p[:2]}"
    print(f"  ✓ {label} DLT  (0,0) → {p[:2].round(3)}, erwartet (10, 5)")

print()
print("TEST 3: Mehr als 4 Punkte (6 Punkte, overdetermined)")
src6 = np.array([[0.,0.],[100.,0.],[100.,100.],[0.,100.],[50.,0.],[50.,100.]])
dst6 = src6 + np.array([10., 5.])
H = normalized_dlt(src6, dst6)
p = H @ np.array([50., 50., 1.]); p /= p[2]
assert np.allclose(p[:2], [60., 55.], atol=0.5)
print(f"  ✓ norm. DLT (50,50) → {p[:2].round(3)}, erwartet (60, 55)")

print()
print("Alle DLT-Tests bestanden ✓")
print("=" * 60)

## Teilaufgabe 1b – Manuelle Korrespondenzpunkte b1 ↔ b2

Zwischen b1.jpg und b2.jpg wurden **24 Punktkorrespondenzen** aus den bekannten Homographien  
von Übung 3 berechnet (Weltkoordinaten-Gitter → Rückprojektion in Pixelraum beider Bilder).

- **4 räumlich verteilte Eckpunkte** (oben-links, unten-links, oben-rechts, unten-rechts) → DLT-Schätzung  
- **20 unabhängige Testpunkte** → Validierung via mittlerem euklidischem Rückprojektionsfehler

Räumliche Verteilung der Trainingspunkte ist entscheidend: kollineare Punkte liefern eine  
schlecht konditionierte Designmatrix und numerisch instabile Homographien.

In [ ]:
ALL_CORR_B1_B2 = [
    ([1293,1101], [326,781]),   ([1289,1347], [292,1057]),   # 0-1:  oben-links
    ([1284,1619], [255,1353]),  ([1278,1921], [216,1672]),   # 2-3
    ([1272,2260], [174,2016]),  ([1265,2643], [128,2389]),   # 4-5:  unten-links
    ([1603,1118], [668,801]),   ([1614,1364], [646,1075]),   # 6-7
    ([1627,1636], [623,1369]),  ([1641,1939], [598,1685]),   # 8-9
    ([1656,2278], [571,2026]),  ([1674,2660], [542,2395]),   # 10-11
    ([1911,1135], [1003,821]),  ([1938,1381], [993,1093]),   # 12-13
    ([1967,1654], [983,1385]),  ([2000,1956], [972,1698]),   # 14-15
    ([2037,2295], [959,2036]),  ([2079,2677], [946,2401]),   # 16-17
    ([2217,1152], [1333,841]),  ([2259,1399], [1334,1111]),  # 18-19: oben-rechts
    ([2306,1671], [1336,1400]), ([2358,1974], [1338,1711]),  # 20-21
    ([2416,2312], [1340,2045]), ([2481,2694], [1342,2407]),  # 22-23: unten-rechts
]

# 4 Eckpunkte (räumlich maximal verteilt) für DLT
TRAIN_IDX = [0, 5, 18, 23]   # oben-links, unten-links, oben-rechts, unten-rechts
TEST_IDX  = [i for i in range(len(ALL_CORR_B1_B2)) if i not in TRAIN_IDX]

src_train = np.array([ALL_CORR_B1_B2[i][0] for i in TRAIN_IDX], dtype=float)
dst_train = np.array([ALL_CORR_B1_B2[i][1] for i in TRAIN_IDX], dtype=float)
src_test  = np.array([ALL_CORR_B1_B2[i][0] for i in TEST_IDX],  dtype=float)
dst_test  = np.array([ALL_CORR_B1_B2[i][1] for i in TEST_IDX],  dtype=float)
src_all   = np.array([c[0] for c in ALL_CORR_B1_B2], dtype=float)
dst_all   = np.array([c[1] for c in ALL_CORR_B1_B2], dtype=float)

print(f"Korrespondenzen gesamt : {len(ALL_CORR_B1_B2)}")
print(f"Training (DLT)         : {len(TRAIN_IDX)} Punkte, Indizes {TRAIN_IDX}")
print(f"Test                   : {len(TEST_IDX)} Punkte")

In [ ]:
def reprojection_error(H, src_pts, dst_pts):
    """Mittlerer euklidischer Abstand zwischen H @ src und dst.

    Returns:
        mean_dist:  skalarer Mittelwert
        dists:      (N,) Array der Einzelabstände
    """
    src_pts = np.array(src_pts, dtype=float)
    dst_pts = np.array(dst_pts, dtype=float)

    src_h = np.hstack([src_pts, np.ones((len(src_pts), 1))]).T  # (3, N)
    proj  = H @ src_h
    proj /= proj[2]
    projected = proj[:2].T                                       # (N, 2)

    dists = np.linalg.norm(projected - dst_pts, axis=1)
    return float(np.mean(dists)), dists

In [ ]:
H_unnorm = dlt(src_train, dst_train)
H_norm   = normalized_dlt(src_train, dst_train)

print("=" * 60)
print("Homographie-Matrizen (geschätzt aus 4 Trainingspunkten)")
print()
print("H_unnorm (unnormierter DLT):")
print(np.round(H_unnorm, 5))
print()
print("H_norm (normierter DLT / Hartley):")
print(np.round(H_norm, 5))
print()
print("=" * 60)
print("Rückprojektionsfehler")
print()
for label, H in [("Unnormierter DLT", H_unnorm), ("Normierter DLT  ", H_norm)]:
    e_tr, _ = reprojection_error(H, src_train, dst_train)
    e_te, _ = reprojection_error(H, src_test,  dst_test)
    print(f"  {label}:")
    print(f"    Train-Fehler (4 Punkte)  : {e_tr:.4f} px")
    print(f"    Test-Fehler  (20 Punkte) : {e_te:.4f} px")
    print()

## Teilaufgabe 1c – Bildregistrierung

Die Homographie $H_{b1 \to b2}$ bildet Pixel aus b1 in das Koordinatensystem von b2 ab.  
Durch **inverse Abbildung** (Rückwärtstransformation) werden für jeden Ausgabepixel  
die Quellkoordinaten in b1 berechnet – so entstehen keine Löcher.

Im Overlay: wo das gewarpte b1 gültige Pixel hat → diese anzeigen, sonst b2.

In [ ]:
def register_images(img_src, img_dst, H):
    """Warpt img_src in das Koordinatensystem von img_dst und erzeugt ein Overlay."""
    output_shape = img_dst.shape[:2]
    warped, mask = projective_transform(img_src, H, output_shape)
    overlay = img_dst.copy()
    overlay[mask > 0] = warped[mask > 0]
    return warped, overlay


IMG_DIR = "../Abgabe3"
img_b1 = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, "b1.jpg")), cv2.COLOR_BGR2RGB)
img_b2 = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, "b2.jpg")), cv2.COLOR_BGR2RGB)

# Homographie aus allen 24 Punkten (höhere Genauigkeit)
H_b1_b2 = normalized_dlt(src_all, dst_all)
err_all, _ = reprojection_error(H_b1_b2, src_all, dst_all)
print(f"H_b1->b2 (norm. DLT, 24 Punkte):")
print(np.round(H_b1_b2, 4))
print(f"Mittlerer Rückprojektionsfehler: {err_all:.4f} px")

warped_b1, overlay = register_images(img_b1, img_b2, H_b1_b2)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, img, title in zip(axes,
    [img_b1, img_b2, overlay],
    ["b1 (Original)", "b2 (Original)", "b1 → b2 (Overlay)"]):
    ax.imshow(img); ax.set_title(title, fontsize=12); ax.axis('off')
plt.suptitle("Bildregistrierung: b1 in b2-Koordinatensystem", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("registration_b1_b2_manual.jpg", dpi=100, bbox_inches='tight')
plt.show()
print("Gespeichert: registration_b1_b2_manual.jpg")

## Teilaufgabe 2a – RANSAC

**RANSAC (Random Sample Consensus)** ist robust gegenüber Ausreißern:

1. **Random Sample**: Wähle zufällig 4 Korrespondenzpaare (Minimalset für DLT)
2. **Modell**: Berechne Homographie mit normiertem DLT
3. **Consensus-Set**: Alle Punkte mit Rückprojektionsfehler $< \varepsilon$ → **Inlier**
4. **Bestes Modell**: Merke das Modell mit den meisten Inliern
5. **Wiederholen** für $N$ Iterationen
6. **Refit**: Abschließend Homographie auf dem vollständigen Consensus-Set neu schätzen

Standard-DLT ist **nicht robust** – ein einziger Ausreißer kann die Schätzung stark verfälschen.

In [ ]:
def ransac(src_pts, dst_pts, n_iter=1000, threshold=5.0, seed=42):
    """RANSAC für Homographie-Schätzung mit normiertem DLT.

    Args:
        src_pts:   (N, 2) Quellpunkte
        dst_pts:   (N, 2) Zielpunkte
        n_iter:    Anzahl RANSAC-Iterationen
        threshold: Inlier-Schwellwert in Pixeln
        seed:      Zufallsseed für Reproduzierbarkeit
    Returns:
        best_H:      (3, 3) Homographie, auf finalem Consensus-Set neu gefittet
        best_inliers: (N,) bool-Maske der Inlier
    """
    src_pts = np.array(src_pts, dtype=float)
    dst_pts = np.array(dst_pts, dtype=float)

    rng = np.random.default_rng(seed)
    best_H, best_inliers = None, np.zeros(len(src_pts), dtype=bool)

    for _ in range(n_iter):
        idx = rng.choice(len(src_pts), 4, replace=False)
        H = normalized_dlt(src_pts[idx], dst_pts[idx])
        _, dists = reprojection_error(H, src_pts, dst_pts)
        inliers = dists < threshold
        if inliers.sum() > best_inliers.sum():
            best_H, best_inliers = H, inliers

    if best_inliers.sum() >= 4:
        best_H = normalized_dlt(src_pts[best_inliers], dst_pts[best_inliers])

    return best_H, best_inliers


# Validierung auf sauberen 24 Punkten
H_ransac_clean, inliers_clean = ransac(src_all, dst_all, n_iter=1000, threshold=5.0)
err_c, _ = reprojection_error(H_ransac_clean, src_all[inliers_clean], dst_all[inliers_clean])
print(f"RANSAC auf 24 sauberen Punkten: {inliers_clean.sum()}/{len(src_all)} Inlier, Fehler = {err_c:.4f} px")

## Teilaufgabe 2b – RANSAC mit Ausreißern

Zu den 24 echten Korrespondenzen werden **6 zufällige Ausreißer** hinzugefügt –  
Punkte mit völlig falschen Zielkoordinaten.

**Erwartetes Ergebnis:**
- DLT auf verrauschten Daten: hoher Fehler (Ausreißer verzerren die Lösung)
- RANSAC: findet korrekte Homographie, alle Ausreißer werden abgelehnt  
- Consensus-Set enthält ausschließlich die 24 echten Korrespondenzen

In [ ]:
rng_out = np.random.default_rng(99)
N_OUTLIERS = 6

outlier_src = rng_out.integers([0,0], [3072,4080], size=(N_OUTLIERS, 2)).astype(float)
outlier_dst = rng_out.integers([0,0], [3072,4080], size=(N_OUTLIERS, 2)).astype(float)

src_noisy = np.vstack([src_all, outlier_src])
dst_noisy = np.vstack([dst_all, outlier_dst])
outlier_idx = set(range(len(src_all), len(src_noisy)))

print(f"Echte Korrespondenzen : {len(src_all)}")
print(f"Ausreißer hinzugefügt : {N_OUTLIERS}")
print(f"Gesamt                : {len(src_noisy)}")

# DLT auf verrauschten Daten (alle 30 Punkte)
H_dlt_noisy = normalized_dlt(src_noisy, dst_noisy)
err_dlt, _  = reprojection_error(H_dlt_noisy, src_all, dst_all)

# DLT nur saubere Punkte (Referenz)
H_dlt_clean = normalized_dlt(src_all, dst_all)
err_clean, _ = reprojection_error(H_dlt_clean, src_all, dst_all)

# RANSAC auf verrauschten Daten
H_ransac, mask = ransac(src_noisy, dst_noisy, n_iter=1000, threshold=5.0, seed=42)
err_ransac_in, _   = reprojection_error(H_ransac, src_noisy[mask], dst_noisy[mask])
err_ransac_all, _  = reprojection_error(H_ransac, src_all, dst_all)
outliers_rejected  = outlier_idx.issubset(set(np.where(~mask)[0].tolist()))

print()
print("=" * 60)
print("Ergebnisse")
print("=" * 60)
print(f"Consensus-Set (RANSAC Inlier) : {mask.sum()} / {len(src_noisy)}")
print(f"Rückprojektionsfehler (Inlier): {err_ransac_in:.4f} px")
print(f"Ausreißer korrekt abgelehnt   : {'✓ Ja' if outliers_rejected else '✗ Nein'}")
print()
print(f"Vergleich – Fehler auf 24 sauberen Punkten:")
print(f"  Norm. DLT (30 Punkte, verunreinigt) : {err_dlt:.4f} px")
print(f"  Norm. DLT (24 Punkte, sauber)       : {err_clean:.4f} px")
print(f"  RANSAC    (30 Punkte, verunreinigt)  : {err_ransac_all:.4f} px")
if err_ransac_all < err_dlt:
    print("  ✓ RANSAC übertrifft DLT auf verrauschten Daten.")

## Teilaufgabe 3a – Automatisches Matching mit SIFT

**SIFT (Scale-Invariant Feature Transform)** detektiert und beschreibt Schlüsselpunkte,  
die invariant gegenüber Skalierung und Rotation sind.

1. `detect_and_compute(gray)` → `(locs, desc)` – locs: (n×4) mit [x,y,scale,angle], desc: (n×128)
2. `match_twosided(desc1, desc2)` → symmetrische Matches (beide Richtungen müssen übereinstimmen),  
   Rückgabe: Array der Länge n_desc1, Wert > 0 = Index in desc2, 0 = kein Match
3. Aus matches src_pts/dst_pts extrahieren für RANSAC

In [ ]:
import sift as sift_module


def sift_correspondences(img1_gray, img2_gray):
    """SIFT detect+compute und beidseitiges Matching.
    Gibt (src_pts, dst_pts, locs1, locs2, matches) zurück.
    """
    locs1, desc1 = sift_module.detect_and_compute(img1_gray)
    locs2, desc2 = sift_module.detect_and_compute(img2_gray)
    matches = sift_module.match_twosided(desc1, desc2)

    idx     = np.where(matches > 0)[0]
    src_pts = locs1[idx, :2].astype(float)
    dst_pts = locs2[matches[idx], :2].astype(float)
    print(f"  {len(src_pts)} symmetrische Matches gefunden")
    return src_pts, dst_pts, locs1, locs2, matches


# Test: b1 und b2
img1_rgb  = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, "b1.jpg")), cv2.COLOR_BGR2RGB)
img2_rgb  = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, "b2.jpg")), cv2.COLOR_BGR2RGB)
img1_gray = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2GRAY)
img2_gray = cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2GRAY)

src_sift, dst_sift, locs1, locs2, matches = sift_correspondences(img1_gray, img2_gray)

plt.figure(figsize=(14, 5))
sift_module.plot_matches(img1_gray, img2_gray, locs1, locs2, matches, show_below=False)
plt.title("SIFT-Matches: b1 → b2", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("sift_matches_b1b2.jpg", dpi=100, bbox_inches='tight')
plt.show()

## Teilaufgabe 3b – Automatische Registrierung aller Bildpaare

Vollständige Pipeline für jedes benachbarte Paar aus Übungsblatt 3:

1. **SIFT Matching** → Korrespondenz-Liste  
2. **RANSAC** (500 Iterationen, Schwelle 5 px) → robuste Homographie  
3. **Registrierung** → Overlay-Visualisierung

Vergleich mit manuell geschätzter Homographie: ähnliche oder gleiche H?

In [ ]:
def auto_register(fname1, fname2, n_iter=500, threshold=5.0):
    """Vollständige SIFT+RANSAC-Pipeline für zwei Bilder."""
    img1 = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, fname1)), cv2.COLOR_BGR2RGB)
    img2 = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, fname2)), cv2.COLOR_BGR2RGB)
    g1   = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
    g2   = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)

    print(f"\n{'='*60}\nPaar: {fname1} → {fname2}")
    src, dst, locs1, locs2, matches = sift_correspondences(g1, g2)

    H, inliers = ransac(src, dst, n_iter=n_iter, threshold=threshold)
    err, _     = reprojection_error(H, src[inliers], dst[inliers])
    print(f"  RANSAC: {inliers.sum()}/{len(src)} Inlier, Fehler = {err:.2f} px")

    _, overlay = register_images(img1, img2, H)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Matches
    axes[0].imshow(sift_module.appendimages(g1, g2), cmap='gray')
    cols1 = img1.shape[1]
    for i, m in enumerate(matches):
        if m > 0:
            axes[0].plot([locs1[i,0], locs2[m,0]+cols1],
                         [locs1[i,1], locs2[m,1]], 'c', lw=0.4, alpha=0.5)
    n1 = fname1.split('.')[0]; n2 = fname2.split('.')[0]
    axes[0].set_title(f"SIFT-Matches: {n1}→{n2}  ({inliers.sum()} Inlier / {len(src)} Matches)",
                      fontsize=11); axes[0].axis('off')

    # Overlay
    axes[1].imshow(overlay)
    axes[1].set_title(f"Registrierung {n1}→{n2}  (Fehler: {err:.2f} px)", fontsize=11)
    axes[1].axis('off')

    plt.tight_layout()
    fig_name = f"registration_{n1}{n2}.jpg"
    plt.savefig(fig_name, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"  Gespeichert: {fig_name}")

    return dict(Paar=f"{n1}→{n2}", Matches=len(src), Inlier=int(inliers.sum()),
                Inlier_pct=f"{100*inliers.sum()/len(src):.1f}%" if len(src) else "N/A",
                Fehler_px=f"{err:.2f}")


PAIRS = [("b1.jpg","b2.jpg"), ("b2.jpg","b3.jpg"), ("b3.jpg","b4.jpg"), ("b4.jpg","b5.jpg")]
summary = [auto_register(f1, f2) for f1, f2 in PAIRS]

print(f"\n{'='*65}")
print("ZUSAMMENFASSUNG – Automatische Registrierung mit SIFT+RANSAC")
print(f"{'='*65}")
print(f"{'Paar':<12} {'Matches':>8} {'Inlier':>8} {'Inlier%':>9} {'Fehler':>10}")
print("-"*65)
for r in summary:
    print(f"{r['Paar']:<12} {r['Matches']:>8} {r['Inlier']:>8} {r['Inlier_pct']:>9} {r['Fehler_px']:>10} px")
print(f"{'='*65}")